# 01 ゲーム理論

**G2：国際紛争の抑止と協調戦略のシミュレーション**（補遺トピック）

- 2 国の協調（C）・非協調（N）を **2×2 の利得行列**で表す
- 4 つの戦略（Always Cooperate・Always Non-cooperate・Tit-for-Tat・Conditional）
- **一回限りのゲーム**（Nash 均衡）と**繰り返しゲーム**（協調率・平均利得）を区別する
- 計算：`game_theory.py`

> **補遺トピック G2・教育用の抽象的な仮想モデル**です。国家（Country A〜E）・利得行列・状態遷移確率・係数はすべて架空で、現実の国際関係・紛争や特定国家の行動を予測するものではありません。扱うのは協調・対話・信頼形成・緊張緩和だけです。

## 準備：実験環境（最初に必ず実行）

G2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                         # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g2-conflict-cooperation-simulation"    # リポジトリの中の G2 フォルダ（ルートが G2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g2_root(p):
    return (Path(p) / "game_theory.py").exists() and (Path(p) / "network_model.py").exists()


def find_g2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G2 のルート
        if is_g2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g2_upload")
    for hit in glob.glob("/content/g2_upload/**/game_theory.py", recursive=True):
        if is_g2_root(Path(hit).parent):
            return Path(hit).parent
    return None


G2_ROOT = find_g2_root()
if G2_ROOT is None:
    raise RuntimeError("G2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G2_ROOT)
if str(G2_ROOT) not in sys.path:
    sys.path.insert(0, str(G2_ROOT))
if not (G2_ROOT / "textbook" / "fixed_stack.py").exists():
    print("⚠️ textbook/ がありません。network_model.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import agents as ag
import game_theory as gt
import trust_model as tm
import state_transitions as st
import simulation as sim
import network_model as nm
import cooperation_policies as cp
import evaluation as ev
import conflict_charts as cc

cc.setup_style()                        # 文字を大きくするなら cc.setup_style(1.3)
COUNTRIES = ag.load_countries()         # 架空の国家 A〜E
PARAMS = sim.load_parameters()          # シミュレーションのパラメータ（すべて架空）
M = gt.load_payoff_matrix()             # 利得行列（架空）
IDS = [c["id"] for c in COUNTRIES]
RUNS = PARAMS["monte_carlo_runs"]       # モンテカルロの反復回数
YEARS = PARAMS["simulation_years"]
LABELS = {s: gt.label(s) for s in ag.STRATEGIES}

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", cc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


SUMMARY_COLUMNS = [("name", "condition", "{}"), ("cooperation_rate", "cooperation", "{:.3f}"),
                   ("cooperation_rate_sd", "sd", "{:.3f}"), ("mutual_cooperation_rate", "mutual coop.", "{:.3f}"),
                   ("cooperation_state_rate", "Cooperation state", "{:.3f}"), ("tension_rate", "Tension", "{:.3f}"),
                   ("conflict_rate", "Conflict", "{:.3f}"), ("conflict_rate_sd", "sd", "{:.3f}")]

print("G2_ROOT:", G2_ROOT)
print("架空の国家:", ", ".join(f"{c['id']}({LABELS[c['strategy']]})" for c in COUNTRIES))
print(f"モンテカルロ {RUNS} 回 × {YEARS} 年、乱数シード {PARAMS['random_seed']}")

## 1. 利得行列（教育用の架空の値）

各セルは（Country A の利得, Country B の利得）。相手が協調しても非協調でも、自分は非協調の方が利得が大きい（**囚人のジレンマ型**）。

In [ ]:
#@title 利得行列のヒートマップ
print("囚人のジレンマ型:", gt.is_prisoners_dilemma(M))
print("一回限りのゲームの Nash 均衡（4 通りの組を全探索）:", gt.nash_equilibria(M))
show(cc.plot_payoff_matrix(M), "01_payoff_matrix")

> **Nash 均衡であること**（一回限りのゲームで、どちらも自分だけ行動を変えても得をしない）と、**長期的に協調が維持されること**は同じ意味ではありません。繰り返しゲームでは、Tit-for-Tat などで協調が続くことがあります。

## 2. 4 つの戦略

| 戦略 | 行動の決め方 |
| --- | --- |
| Always Cooperate | 常に協調 |
| Always Non-cooperate | 常に非協調 |
| Tit-for-Tat | 初回は協調、以後は相手の前回の行動をまねる |
| Conditional | 協調傾向・信頼度・相手の最近の協調率・経済的相互依存で決まる確率で協調（上限 0.95：信頼が高くても必ず協調するとは限らない） |

どの戦略にも、確率 `action_noise`（2%）の**行動の誤り**があります。

## 3. 繰り返しゲーム（同じ相手・同じ初期条件）

4 つの戦略を、同じ相手（Tit-for-Tat、または Conditional）と 50 回対戦させます。

In [ ]:
ROUNDS = 50
results = {}
for opp_strategy in ["tit_for_tat", "conditional"]:
    opp = ag.make_country("O", 0.6, 0.5, 0.5, opp_strategy)
    players = [(s, ag.make_country("S", 0.6, 0.5, 0.5, s)) for s in ag.STRATEGIES]
    results[opp_strategy] = gt.tournament(players, ROUNDS, M, PARAMS, PARAMS["random_seed"], opponent=opp)
rows = [{"name": f"{LABELS[s]} vs {LABELS[o]}", **r[s]} for o, r in results.items() for s in ag.STRATEGIES]
table(rows, [("name", "match", "{}"), ("first_round_payoff", "1st payoff", "{:.1f}"),
             ("average_payoff", "avg payoff", "{:.2f}"), ("cumulative_payoff", "cumulative", "{:.0f}"),
             ("cooperation_rate", "cooperation", "{:.2f}"), ("mutual_cooperation_rate", "mutual coop.", "{:.2f}"),
             ("noncooperation_rate", "non-coop.", "{:.2f}")])

In [ ]:
#@title 戦略別の平均利得と協調率
names = [LABELS[s] for s in ag.STRATEGIES]
show(cc.plot_grouped(names, ["vs Tit-for-Tat", "vs Conditional"],
                     {"vs Tit-for-Tat": [results["tit_for_tat"][s]["average_payoff"] for s in ag.STRATEGIES],
                      "vs Conditional": [results["conditional"][s]["average_payoff"] for s in ag.STRATEGIES]},
                     "Average payoff per round by strategy (fictional payoffs)", "average payoff",
                     colors={"vs Tit-for-Tat": "#1baf7a", "vs Conditional": "#6250d6"}, ylim=(0, 5.5)),
     "01_payoff_by_strategy")
show(cc.plot_grouped(names, ["vs Tit-for-Tat", "vs Conditional"],
                     {"vs Tit-for-Tat": [results["tit_for_tat"][s]["cooperation_rate"] for s in ag.STRATEGIES],
                      "vs Conditional": [results["conditional"][s]["cooperation_rate"] for s in ag.STRATEGIES]},
                     "Cooperation rate by strategy", "cooperation rate",
                     colors={"vs Tit-for-Tat": "#1baf7a", "vs Conditional": "#6250d6"}), "01_coop_by_strategy")

## 4. 短期の利得と長期の利得（RQ5）

1 回目の利得（短期）と、50 回の平均利得（長期）を比べます。

In [ ]:
r = results["tit_for_tat"]
for s, v in gt.rank({s: r[s]["first_round_payoff"] for s in ag.STRATEGIES}):
    print(f"1 回目の利得 {v:.1f}  50 回の平均利得 {r[s]['average_payoff']:.2f}  {LABELS[s]}")
print("50 回の平均利得の順位:", " > ".join(LABELS[s] for s, _v in gt.rank({s: r[s]["average_payoff"] for s in ag.STRATEGIES})))

## 5. 学生向け TODO

In [ ]:
# TODO 1：利得行列を変更する（例：協調の利得を 4 にする / 一方的な非協調の利得を 3 にする）
M2 = {"C": {"C": 4, "N": 0}, "N": {"C": 3, "N": 1}}
print("Nash 均衡:", gt.nash_equilibria(M2), " 囚人のジレンマ型:", gt.is_prisoners_dilemma(M2))
# TODO 2：戦略を変更する（data/sample_countries.json、または ag.make_country の strategy）
a = ag.make_country("A", 0.6, 0.5, 0.5, "conditional")
b = ag.make_country("B", 0.6, 0.5, 0.5, "tit_for_tat")
# TODO 3：結果を比較する
for name, mat in [("original", M), ("modified", M2)]:
    g = gt.play_repeated(a, b, 50, mat, PARAMS, seed=PARAMS["random_seed"])
    print(f"{name:9}: 協調率 {g['cooperation_rate']:.2f}  相互協調率 {g['mutual_cooperation_rate']:.2f}  平均利得 {g['average_payoff']:.2f}")

## 考察（チームで記入）

- **TODO**：一回限りのゲームの Nash 均衡と、繰り返しゲームでの協調率の違いを説明する
- **TODO**：1 回目の利得が大きい戦略は、長期の平均利得も大きかったか（RQ5）
- **TODO**：この利得行列で表せない国家間関係の要素は何か（モデルの限界）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 スタック・キュー、第5章 再帰、第6章 単純挿入ソート）
- R. Axelrod, *The Evolution of Cooperation*, Basic Books, 1984（繰り返し囚人のジレンマと Tit-for-Tat）
- G2 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_countries.json`、`data/payoff_matrix.json`、`data/simulation_parameters.json`（この教材のための**架空のデータ**）